<a href="https://colab.research.google.com/github/astha171206/My-colab-project-/blob/main/project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

In [ ]:
dataset=load_dataset("CShorten/ML-ArXiv-Papers",split='train')

In [ ]:
print(dataset)

In [ ]:
dataset[0]

In [ ]:
import pandas as pd

In [ ]:
df=pd.DataFrame(dataset)
df

In [ ]:
df.columns

In [ ]:
df=df[['title','abstract']]

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df=df.head(15000)

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df["paper_text"]=df["title"]+" "+df["abstract"]

In [ ]:
df[["paper_text"]].head()

In [ ]:
type(df["paper_text"])

In [ ]:
df[["paper_text"]].head()

In [ ]:
print(df["paper_text"].iloc[0])

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
print(type(model))

In [ ]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

In [ ]:
sample_text=df["paper_text"].iloc[0]
sample_text

In [ ]:
embedding=model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

In [ ]:
embedding[:56]

In [ ]:
sample_embedding=model.encode(df["paper_text"].head(5).to_list())

In [ ]:
print(sample_embedding.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
similarity=cosine_similarity(sample_embedding[0].reshape(1, -1),sample_embedding[0].reshape(1, -1))
print(similarity)

In [ ]:
similarity=cosine_similarity(sample_embedding[0].reshape(1, -1),sample_embedding[1].reshape(1, -1))
print(similarity)

In [ ]:
for i in range(1,5):
  sim=cosine_similarity(sample_embedding[0].reshape(1, -1),sample_embedding[i].reshape(1, -1))
  print(sim)


**Generate Full Embedding**

In [ ]:
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save("paper_embeddings.npy", embeddings)
    print("Embeddings saved successfully!")

In [ ]:
#embedding=model.encode(df["paper_text"].to_list(),batch_size=32,show_progress_bar=True)

In [ ]:
print(embedding.shape)
print(type(embedding))

In [ ]:
embedding.dtype

In [ ]:
!pip install faiss_cpu

In [ ]:
import faiss

In [ ]:
faiss.normalize_L2(embeddings)

In [ ]:
index = faiss.IndexFlatIP(384)

In [ ]:
index.add(embeddings)

In [ ]:
print(index.ntotal)

In [ ]:
if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)
    faiss.write_index(index, "paper_faiss.index")
    print("FAISS index saved successfully!")

In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape

In [ ]:
faiss.normalize_L2(query_embedding)

In [ ]:
D,I=index.search(query_embedding,5)
print(D)
print(I)

In [ ]:
print(df.iloc[10466]["title"])

In [ ]:
print(df.iloc[10466]["abstract"])

In [ ]:
print(df.iloc[11873]["title"])

In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  return D,I

In [ ]:
D,I=search_paper("deep learning for medical image analysis")
print(D)
print(I)

In [ ]:
def search_paper(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Title",df.iloc[idx]["abstract"][:500])
    print()

In [ ]:
search_paper("deep learning for medical image analysis")

In [ ]:
!pip install transformers==4.46.3

In [ ]:
from transformers import pipeline
summarizer=pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

In [ ]:
type(summarizer)

In [ ]:
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)

In [ ]:
type(summary)

In [ ]:
type(summary[0])

In [ ]:
summary[0]['summary_text']

In [ ]:
for score,idx in zip(D[0],I[0]):
    print("similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
    print(summary)
    print(summary[0]['summary_text'])
    print()

In [ ]:
def search_paper_summarizer(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Title",df.iloc[idx]["abstract"][:500])
    print()

    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
    print(summary)
    print(summary[0]['summary_text'])
    print()

In [ ]:
search_paper_summarizer('Deep learning in medical imaging' ,k=5)

In [ ]:
pip install keybert==0.8.5

In [ ]:
from keybert import KeyBERT

In [ ]:
kw_model=KeyBERT(model)

In [ ]:
type(kw_model)#model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
print(df.iloc[10466]["abstract"])

In [ ]:
text=df.iloc[10466]["abstract"]
keywords=kw_model.extract_keywords(text)

In [ ]:
print(keywords)

In [ ]:
print(type(keywords))

In [ ]:
print(type(keywords[0]))

In [ ]:
keywords=kw_model.extract_keywords(text,keyphrase_ngram_range=(1,3),stop_words='english')

In [ ]:
print(keywords)

In [ ]:
def search_paper_summarizer(query,k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("similarity score",score)
    print("Title",df.iloc[idx]["title"])
    print("Title",df.iloc[idx]["abstract"][:500])
    print()

    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40,do_sample=False)
    print(summary)
    print(summary[0]['summary_text'])
    print()
    keywords=kw_model.extract_keywords(text,keyphrase_ngram_range=(1,3),stop_words='english')
    print(keywords)
    for k,s in keywords:
      print(k)

**Abstract summarization**

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm



In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
def get_entities(text):
    doc = nlp(text)
    entities_found = []

    for word in doc.ents:
        if word.label_ in ["ORG", "PRODUCT", "DATE"]:
            entities_found.append(f"{word.text} ({word.label_})")

    return entities_found

In [ ]:
paper_text = "We used PyTorch to build a BERT model in 2023."
paper_entities = get_entities(paper_text)
print("Found Entities:", paper_entities)

In [ ]:
pip install langchain-community langchain-core

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def search_paper(query:str)->str:
  "search from the faiss database and return the most relevant result"

In [ ]:
pip install langchain-community langchain-core langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFacePipeline

In [ ]:
llm=HuggingFacePipeline(pipeline=summarizer)

In [ ]:
llm

In [ ]:
response=llm.invoke("Although depressive symptoms (DS) and physical activity (PA) have each been associated with social frailty (SF), their combined associations remain unclear. This study examined the associations of DS and objectively measured PA with SF in community-dwelling older adults. This cross-sectional study included 147 adults aged ≥ 65 years. DS were assessed using the Japanese version of the 15-item Geriatric Depression Scale, and PA was measured objectively via accelerometer-based daily step counts, classified as < 4,000 or ≥ 4,000 steps/day. SF was evaluated using the five-item criteria proposed by Makizako et al. A two-way analysis of covariance was performed with DS (yes/no) and step count (≥ 4,000/<4,000) as independent variables and the total number of positive SF components (0–5) as the dependent variable, adjusting for relevant covariates. Participants with DS and those walking < 4,000 steps/day had significantly higher SF scores (both P < 0.001). A significant interaction between DS and step count was also observed (P = 0.006); however, this finding should be interpreted cautiously because the subgroup with both DS and low step counts was small (n = 6). These findings suggest that both DS and low PA are associated with higher SF scores, while the potential combined association of these factors warrants further investigation in larger studies.")

In [ ]:
response

In [ ]:
print(type(response))